# Pilot Re-Score Launcher — `faithful-ids` (current extractor instrument, token-free replay)

Scores runs with whatever extractor version is on the checked-out ref. The
`extractor=` stamp each results block prints (read from the run's own
`claims.jsonl`) is authoritative — 1.2.0 as of commit `25812ee`.

**Purpose.** Re-score the three completed pilot runs (Qwen2.5-3B, Mistral-7B, Qwen3-8B) with the
**Pass B** extractor fix (**instrument v1.0.0 → 1.1.0**) **without spending any GPU or LLM tokens.**
Each generation is *replayed* from that run's cached LLM ledger; only the detector, TreeSHAP,
extraction, and metrics are recomputed — under the corrected instrument.

**What Pass B fixed (why the numbers move):**
- **Signed-number direction parsing → DSA.** A raw-SHAP dump (`Feature=-7.98`, no direction word)
  used to default to POSITIVE; it now reads the sign. **Litmus: b0 DSA 0.636 → ~1.0.**
- **Longest-match residual-span guard → precision / F1 / ARC / HFR.** Substring feature collisions
  (`Packet Length Mean` ⊂ `Fwd Packet Length Mean`) no longer double-match.

The explanations are **identical** to the original runs (replay) — this is an *instrument-only*
re-score. It revalidates the faithfulness story on a correct extractor, and resolves whether
Mistral's **b4 dip** was real or an extraction artifact.

## Re-score · Qwen3-32B (N=60 smoke)

Ledger comes from the 32B pilot export: copy `pilot_artifacts/runs/_pilot_llm_cache/`
into the ledgers dataset as `qwen3_32b_4bit/` (so `ledger.jsonl` sits at
`<LEDGER_BASE>/qwen3_32b_4bit/ledger.jsonl`). N=60 is set automatically from `MODEL_N`.
Under extractor 1.4.0 expect the B3 paraphrase zeros (38/60) to disappear and
mention to recover toward ~0.9; b0/b1 must stay exactly 1.000.


In [ ]:
rescore('qwen3_32b_4bit')


## How to use — read once

1. **Run the Setup section once** (clone → install → config/data → gates → helpers).
2. For each model, run its **one** re-score cell (3B / Mistral / Qwen3). It does everything:
   replay → analysis → figures/tables → prints the results → exports a zip.
3. After all three, run **Compare all three** for the corrected cross-model table.

**Rules of the road**
- ⛔ **This notebook never generates** — it is replay-only (no tokens). For a fresh *live* run use
  `kaggle_pilot_launcher.ipynb` instead.
- Analysis reads `source.select: latest`, so each model cell analyzes the run it just made. Running
  all three in one session leaves 3 runs in `runs/` — the **Compare** cell reads all of them.
- ✅ **Litmus:** each results block prints **b0 DSA** — it must be **~1.0** (was 0.636). That proves
  the 1.1.0 extractor scored the run.
- If a re-score prints **`ReplayMiss`**, the CICIDS input samples different instances than the
  original run → use the *same* dataset the originals used (the retrained detector should hash to
  `xgboost@59b0ad0d`).

**Prerequisites — add as Inputs (right sidebar → Add Input):**
- **Ledgers dataset**, one folder per model each holding `ledger.jsonl`:
  `…/faithful-ids-ledgers/{qwen2_5_3b_instruct, mistral_7b_instruct_4bit, qwen3_8b_4bit}/ledger.jsonl`
  → set its base path in the **Setup · config** cell (`LEDGER_BASE`).
- **CICIDS2017** CSVs (same data the original runs used) — auto-detected below.

## Setup · 1 — clone the repo & install (light: no torch/bitsandbytes; replay loads no model)

In [ ]:
%%bash
set -e
rm -rf /kaggle/working/faithful-ids
git clone --quiet https://github.com/SLIMIHamda/faithful-ids.git /kaggle/working/faithful-ids
cd /kaggle/working/faithful-ids && git checkout --quiet main
pip install -q -e . --no-deps 2>&1 | tail -n 1
pip install -q pyyaml jsonschema import-linter 2>&1 | tail -n 1
echo "commit: $(git rev-parse --short HEAD)   (want 9dbd91d or later)"
python -c "import xgboost, shap; print('xgboost', xgboost.__version__, '| shap', shap.__version__)" 

## Setup · 2 — config & locate data

`LEDGER_BASE` is the **only** path you may need to edit. The run invariants (`N`, `MAX_ROWS`) are
fixed here and **must match the original runs**, or replay will cache-miss.

In [ ]:
import os, glob
# --- EDIT ONLY IF your ledger dataset path differs -------------------------
LEDGER_BASE = '/kaggle/input/datasets/slimihamda/faithful-ids-ledgers'
# ---------------------------------------------------------------------------
os.environ['FAITHFULIDS_REF']                = 'main'
# N MUST match each original run; set per-model by rescore() from MODEL_N below.
MODEL_N = {'qwen2_5_3b_instruct': '150', 'mistral_7b_instruct_4bit': '150',
           'qwen3_8b_4bit': '150', 'qwen3_32b_4bit': '60'}
os.environ['FAITHFULIDS_MAX_ROWS']           = '200000'   # MUST match
os.environ['FAITHFULIDS_ENFORCE_COMPETENCE'] = '0'        # report, don't halt (pilot)

# Auto-detect a CICIDS2017 CSV directory (override by setting CIC_DIR yourself)
CIC_DIR = os.environ.get('CIC_DIR')
if not CIC_DIR:
    cands = sorted({os.path.dirname(p) for p in glob.glob('/kaggle/input/**/*.csv', recursive=True)})
    def _has_label(d):
        f = sorted(glob.glob(os.path.join(d, '*.csv')))[:1]
        return f and 'label' in open(f[0]).readline().lower()
    cands = [d for d in cands if _has_label(d)] or cands
    CIC_DIR = cands[0] if cands else ''
assert CIC_DIR, 'No CICIDS CSVs found under /kaggle/input — add the dataset as Input.'
os.environ['FAITHFULIDS_DATA_DIR'] = CIC_DIR
print('data dir   :', CIC_DIR)
print('ledger base:', LEDGER_BASE, '| exists:', os.path.isdir(LEDGER_BASE))
print('N per model =', MODEL_N, '| MAX_ROWS =', os.environ['FAITHFULIDS_MAX_ROWS'])

## Setup · 3 — reproducibility gates (optional sanity check)

In [ ]:
%%bash
cd /kaggle/working/faithful-ids
echo '== firewall =='; python tools/firewall_check.py
echo '== validate configs =='; python -m faithfulids.orchestration.validate_configs

## Setup · 4 — helpers (defines `rescore(model)` and `compare_runs()`)

`rescore(model)` sets that model's env (`FAITHFULIDS_PILOT_LLM`, `FAITHFULIDS_LLM_CACHE_DIR`),
runs the replay re-score + analysis + figures/tables, prints the Layer-1 table **with the DSA
litmus**, and exports `/kaggle/working/rescore_<model>.zip`. No parameters to edit per model.

In [ ]:
import os, glob, json, subprocess, statistics
REPO = '/kaggle/working/faithful-ids'
MODELS = ['qwen2_5_3b_instruct', 'mistral_7b_instruct_4bit', 'qwen3_8b_4bit',
          'qwen3_32b_4bit']
GENS = ['b0_raw_shap', 'b1_template', 'b2_zeroshot', 'b3_dte_style', 'b4_vte']

def _sh(cmd):
    print('$', cmd, flush=True)
    return subprocess.run(cmd, shell=True, cwd=REPO).returncode

def _latest_run():
    runs = sorted(glob.glob(f'{REPO}/runs/EXP-PILOT-001/EXP-PILOT-001__*'))
    return runs[-1] if runs else None

def _by_inst(rows, g, metric):
    return {r['instance_id']: r['value'] for r in rows
            if r['layer'] == 'layer1' and r['metric'] == metric
            and r.get('grouping', {}).get('generator_id') == g}

def _l1(rows, g, metric):
    v = list(_by_inst(rows, g, metric).values())
    return statistics.mean(v) if v else float('nan')

def _l1_gated(rows, g, metric='dsa_asserted', gate='direction_assertion_rate', gate_min=None):
    """Mean of `metric` over instances where the gate metric passes — NaN-exclusion
    (2026-07-14): the target is UNDEFINED (structural 0.0) on those instances. Two
    uses: dsa_asserted gated on direction_assertion_rate>0 (default), and arc gated
    on arc_n_pairs>=2 (gate_min=2). Mirrors analysis.run._instance_values, so these
    in-notebook numbers match analysis/outputs/pilot_dsa_asserted and pilot_arc."""
    vals, grate = _by_inst(rows, g, metric), _by_inst(rows, g, gate)
    keep = (lambda x: x >= gate_min) if gate_min is not None else (lambda x: x > 0.0)
    kept = [v for inst, v in vals.items() if keep(grate.get(inst, 0.0))]
    return statistics.mean(kept) if kept else float('nan')

def show_latest():
    import yaml
    rd = _latest_run()
    if not rd: print('no run found'); return
    cfg = yaml.safe_load(open(f'{rd}/config.resolved.yaml'))
    rows = [json.loads(l) for l in open(f'{rd}/artifacts/metrics.jsonl')]
    ev = json.loads(open(f'{rd}/artifacts/claims.jsonl').readline())['extractor_version']
    print(f"\nrun {os.path.basename(rd)} | llm={cfg['llm']} | mode={cfg.get('llm_mode')} | extractor={ev}")
    # dsa = descriptive (includes defaults); dsa_a = confirmatory dsa_asserted gated on
    # arate>0; arate = direction_assertion_rate; ARC gated on arc_n_pairs>=2. Read the
    # gated columns WITH their coverage (arate / pair count), never alone.
    print(f"{'generator':14s}{'mention_f1':>12s}{'DSA':>8s}{'dsa_a':>8s}{'arate':>8s}{'ARC':>8s}")
    for g in GENS:
        print(f"{g:14s}{_l1(rows,g,'mention_f1'):12.3f}{_l1(rows,g,'dsa'):8.3f}"
              f"{_l1_gated(rows,g):8.3f}{_l1(rows,g,'direction_assertion_rate'):8.3f}"
              f"{_l1_gated(rows,g,'arc','arc_n_pairs',gate_min=2):8.3f}")
    b0, b0a = _l1(rows, 'b0_raw_shap', 'dsa'), _l1_gated(rows, 'b0_raw_shap')
    flag = 'OK' if b0 > 0.95 else 'CHECK — should be ~1.0'
    print(f"\nLITMUS  b0 DSA = {b0:.3f} | b0 dsa_asserted = {b0a:.3f}  [{flag}]  (was 0.636 under 1.0.0)")

def rescore(model):
    assert model in MODELS, f'unknown model {model!r}; pick one of {MODELS}'
    cache = f'{LEDGER_BASE}/{model}'
    ledger = f'{cache}/ledger.jsonl'
    assert os.path.isfile(ledger), (f'ledger not found: {ledger}\n'
        'Check LEDGER_BASE (Setup·2) and that the ledgers dataset is added as an Input.')
    os.environ['FAITHFULIDS_PILOT_LLM'] = model
    os.environ['FAITHFULIDS_PILOT_N'] = MODEL_N[model]  # replay hard-errors on N mismatch
    os.environ['FAITHFULIDS_LLM_CACHE_DIR'] = cache
    n = sum(1 for _ in open(ledger))
    print(f'== re-scoring {model} ==  ledger {ledger} ({n} cached calls)\n')
    if _sh('python tools/rescore_run.py --experiment EXP-PILOT-001') != 0:
        print('\n!! rescore FAILED (see error above) — not exporting'); return
    for cfg in ('pilot_faithfulness', 'pilot_coverage', 'pilot_dsa_asserted', 'pilot_arc'):
        _sh(f'python -m analysis.run --config {cfg}')
    _sh('python -m paper.figures.build'); _sh('python -m paper.tables.build')
    show_latest()
    _sh(f'rm -rf /kaggle/working/rescore_{model} && mkdir -p /kaggle/working/rescore_{model} && '
        f'cp -r runs analysis/outputs paper/figures/generated paper/tables/generated '
        f'/kaggle/working/rescore_{model}/ 2>/dev/null; '
        f'cd /kaggle/working && zip -qr rescore_{model}.zip rescore_{model} && ls -la rescore_{model}.zip')

def compare_runs():
    """model x generator table over every run present in this session."""
    import yaml
    by, evs = {}, set()
    for rd in sorted(glob.glob(f'{REPO}/runs/EXP-PILOT-001/EXP-PILOT-001__*')):
        cfg = yaml.safe_load(open(f'{rd}/config.resolved.yaml'))
        by[cfg['llm']] = [json.loads(l) for l in open(f'{rd}/artifacts/metrics.jsonl')]
        evs.add(json.loads(open(f'{rd}/artifacts/claims.jsonl').readline())['extractor_version'])
    if not by: print('no runs yet — run the model cells first'); return
    ev = ' / '.join(sorted(evs)) + (' — MIXED VERSIONS, do not tabulate together!' if len(evs) > 1 else '')
    order = [m for m in MODELS if m in by]
    # dsa_asserted gated on arate>0, arc gated on arc_n_pairs>=2 (NaN-exclusion);
    # mention_f1 / dsa are plain means.
    for metric in ('mention_f1', 'dsa', 'dsa_asserted', 'arc'):
        gated = metric in ('dsa_asserted', 'arc')
        note = ' (gated)' if gated else ''
        print(f'\n=== Layer-1 {metric}{note} (extractor {ev}) ===')
        print(f"{'generator':14s}" + ''.join(f'{m[:12]:>14s}' for m in order))
        for g in GENS:
            if metric == 'dsa_asserted':
                cell = lambda m: _l1_gated(by[m], g)
            elif metric == 'arc':
                cell = lambda m: _l1_gated(by[m], g, 'arc', 'arc_n_pairs', gate_min=2)
            else:
                cell = lambda m: _l1(by[m], g, metric)
            print(f"{g:14s}" + ''.join(f'{cell(m):14.3f}' for m in order))
print('helpers ready: rescore(<model>), compare_runs(), show_latest()')

## Re-score — Qwen2.5-3B (baseline, fp16)

The original pilot generator. Expect b0 DSA to jump to ~1.0 and b3's 66 catastrophic zeros to be **unaffected** by the extractor (they were a model failure, not a parser bug).

*Runs replay → analysis → figures → results → export `rescore_qwen2_5_3b_instruct.zip`. No parameters to edit.*

In [ ]:
rescore('qwen2_5_3b_instruct')

## Re-score — Mistral-7B (4-bit)

Different family, ~7B. Watch b4 here — the re-score decides whether Mistral's **b4 dip** (0.639, below 3B) was real or an artifact of the old sign/span parsing.

*Runs replay → analysis → figures → results → export `rescore_mistral_7b_instruct_4bit.zip`. No parameters to edit.*

In [ ]:
rescore('mistral_7b_instruct_4bit')

## Re-score — Qwen3-8B (4-bit, thinking disabled)

Newest + biggest. Expect the largest DSA/ARC corrections and the cleanest b3 (~0.89).

*Runs replay → analysis → figures → results → export `rescore_qwen3_8b_4bit.zip`. No parameters to edit.*

In [ ]:
rescore('qwen3_8b_4bit')

## Compare all three (run after the three cells above)

Corrected cross-model table — mention_f1 and DSA per generator, straight from each run's
`metrics.jsonl`. This is the instrument-fixed version of the three-run comparison.

In [ ]:
compare_runs()

## Notes & troubleshooting

- **`ledger not found …`** → `LEDGER_BASE` (Setup·2) is wrong or the ledgers dataset isn't added as
  an Input. Fix the path, re-run Setup·2, re-run the model cell.
- **`ReplayMiss` mid-run** → the CICIDS input differs from the original run's data. Use the same
  dataset; the retrained detector should hash to `xgboost@59b0ad0d`.
- **b0 DSA not ~1.0** → the repo isn't at the Pass B commit. Re-run Setup·1 (needs `9dbd91d`+).
- **Downloads** → `rescore_<model>.zip` are written to `/kaggle/working/` (Output tab). Each holds
  that model's `runs/`, analysis outputs, and figures/tables.
- **This is NON-CITABLE** (ref=`main`, pilot tier). For a citable re-score, tag the commit and set
  `FAITHFULIDS_REF` to the tag in Setup·2.